In [ ]:
import argparse
import json
from transformers import AutoTokenizer
from tqdm import tqdm
import re
import numpy as np
from collections import defaultdict
from sklearn.mixture import GaussianMixture
from math import sqrt, log
from collections import Counter
from scipy import integrate

In [ ]:
model_flops = {
    'meta-llama/Llama-3.2-3B-Instruct': 3000000000,
    'Qwen/Qwen2.5-3B-Instruct': 3000000000,
    'google/gemma-3-4b-it': 4000000000,
    'Qwen/Qwen2.5-7B-Instruct': 7000000000,
    'google/gemma-3-27b-it': 27000000000,
}

dataset = "math"
# model = 'meta-llama/Llama-3.2-3B-Instruct'
model = "Qwen/Qwen2.5-7B-Instruct"
# model = 'google/gemma-3-27b-it'
posterior_threshold = 0.90
beta_threshold = 0.95
calibration_size = 128
self_certainty = 0

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)
print(model, "ready!")

# load data
if dataset == 'omnimath':
    with open(f"./logs/self_certainty/sc_16_{dataset}_2048_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
else:
    with open(f"./logs/self_certainty/sc_16_{dataset}_1024_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

In [ ]:
def gaussian_pdf(s, mu, sigma):
    return (1.0 / (np.sqrt(2*np.pi) * sigma)) * np.exp(-(s - mu)**2 / (2 * sigma**2))

def posterior_correct(s, mu_c, sigma_c, w_c, mu_w, sigma_w, w_w):
    """
    P(correct | s)
    """
    num = w_c * gaussian_pdf(s, mu_c, sigma_c)
    den = num + w_w * gaussian_pdf(s, mu_w, sigma_w)
    return num / den

def is_correct_with_threshold(s, mu_c, sigma_c, w_c, mu_w, sigma_w, w_w, q=0.95):
    """
    P(correct | s) >= q 인지 True/False 반환
    """
    return posterior_correct(s, mu_c, sigma_c, w_c, mu_w, sigma_w, w_w) >= q


def apply_chat_template(dataset, question, response):
    if dataset == "gpqa_diamond" or dataset == "arcChallenge":
        prompt = f"{question}\n\nBased on the above, what is the single, most likely answer choice? Answer in the format \"The correct answer is (insert answer here)\"."
        chat_template = [{'role': 'user', 'content': prompt}, {"role": "assistant", "content": response}]
    else:
        chat_template = [{'role': 'user', 'content': question}, {"role": "assistant", "content": response}]

    return tokenizer.apply_chat_template(chat_template, tokenize=False, add_generation_prompt=False)

def extract_boxed_content(text):
    start = text.find(r"\boxed{")
    if start == -1:
        return None
    i = start + len(r"\boxed{")
    depth = 1
    content = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        content.append(c)
        i += 1
    return "".join(content)

def clean(value):
    if dataset == 'gsm8k':
        value = value.replace(',','')
        numbers = re.findall(r"\d+(?:\.\d+)?", value)
        if len(numbers) > 0:
            value = numbers[-1]
        else:
            value= None
    else:
        final_value = extract_boxed_content(value)
        if final_value is None:
            value = ""
        else:
            value = final_value.replace(' ','')
    return value

In [ ]:
# compute self_certainty_variant
for inst in tqdm(data):
    sample = inst['self_certainty_per_token']
    responseLen = len(tokenizer.encode(inst['response'], add_special_tokens=False))
    # sliding_window_size = 64
    sliding_window_size = 128
    clusters = []
    for i in range(0, len(sample)):
        window = sample[i:i+sliding_window_size]
        if window:
            cluster_avg = sum(window) / len(window)
            clusters.append(cluster_avg)
    # tail confidence
    lowest_group_confidence = min(clusters)
    avg_group_confidence = sum(clusters) / len(clusters)
    Bottom10group_confidence = sum(sorted(clusters)[:max(1, len(clusters)//10)]) / max(1, len(clusters)//10)
    inst['lowest_group_confidence'] = lowest_group_confidence
    inst['avg_group_confidence'] = avg_group_confidence
    inst['Bottom10group_confidence'] = Bottom10group_confidence




# compute predefined confidence mean and std
predefined_confidences = []
question_answer = {}
question_set=set()
for inst in data:
    question = inst['question']
    response = inst['response']
    if question not in question_answer:
        question_answer[question] = inst['answer']
    if self_certainty:
        internal_value = inst['self_certainty_overall']
    else:
        internal_value = inst['Bottom10group_confidence']
    if question not in question_set:
        predefined_confidences.append(internal_value)
        question_set.add(question)
print(f"Predefined confidence mean: {np.mean(predefined_confidences)}")
predefined_confidence_mean = np.mean(predefined_confidences)
predefined_confidence_std = np.std(predefined_confidences)
len(predefined_confidences)


# compute gmm for predefined confidence
X = np.array(predefined_confidences).reshape(-1, 1)

# Fit GMM with 2 components
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=0)
gmm.fit(X)

means = gmm.means_.flatten()
vars_ = gmm.covariances_.flatten()
weights = gmm.weights_.flatten()
print("Means:", means)
print("Variances:", vars_)
print("Weights:", weights)

# Sort by mean
idx = np.argsort(means)
means, vars_, weights = means[idx], vars_[idx], weights[idx]

# Intersection point
mu1, mu2 = means
s1, s2 = sqrt(vars_[0]), sqrt(vars_[1])
w1, w2 = weights

if mu1 >= mu2:
    mu_c, s_c, w_c = mu1, s1, w1
    mu_w, s_w, w_w = mu2, s2, w2
else:
    mu_c, s_c, w_c = mu2, s2, w2
    mu_w, s_w, w_w = mu1, s1, w1


plist = np.sort(np.asarray(predefined_confidences, float))
tau = None
for i, p in enumerate(plist):
    s = p
    q= posterior_threshold
    posterior = posterior_correct(s, mu_c, s_c, w_c, mu_w, s_w, w_w)
    accept = is_correct_with_threshold(s, mu_c, s_c, w_c, mu_w, s_w, w_w, q)
    if accept:
        print("P(correct|s) =", posterior)
        print("threshold: ", s)
        print("idx: ",i)
        tau = s
        break
if tau is None:
    tau = plist[-1]
tau = float(tau)
print("tau: ", tau)
predefined_threshold = max(mu_c, tau)



In [ ]:
results = defaultdict(list)
error_cnt=0
for inst in data:
    question = inst['question']


    if dataset == 'gsm8k':
        pred = inst['pred'].replace(' ','').strip()
        verdict = clean(inst['pred']) == clean(inst['answer'])
    else:
        pred = clean(inst['response'])
        if pred == '':
            pred = clean(inst['pred'])
            if pred == '':
                pred = inst['pred'].replace(' ','').strip()
        
        if pred == '':
            error_cnt += 1
        verdict = inst['verdict']

    response = inst['response']
    if pred == '':
        error_cnt+=1
    
    if self_certainty:
        internal_value = inst['self_certainty_overall']
    else:
        internal_value = inst['Bottom10group_confidence']
    
    
    if question not in results:
        results[question] = []
    
    if dataset == 'gsm8k':
        results[question].append((clean(pred), internal_value, verdict, apply_chat_template(dataset, question, response)))
    else:
        results[question].append((pred, internal_value, verdict, apply_chat_template(dataset, question, response)))


# log_predefined_confidences mean and std
log_predefined_confidences = np.log(np.array(predefined_confidences))
log_predefined_confidence_mean = np.mean(log_predefined_confidences)
log_predefined_confidence_std = np.std(log_predefined_confidences)

In [ ]:
import numpy as np


# def sigmoid(Z):
#     """
#     Implements the sigmoid activation in bumpy

#     Arguments:
#     Z -- numpy array of any shape

#     Returns:
#     A -- output of sigmoid(z), same shape as Z
#     cache -- returns Z, useful during backpropagation
#     """

#     cache=Z
#     A=1/(1+(np.exp((-Z))))

#     return A, cache


correct_updates = []
wrong_updates = []
lambda_value=0.7

majority_voting_results = defaultdict(list)
for question in tqdm(results):
    majority_voting_results[question] = []
    # for i in range(len(results[question])):
    for i in range(16):
        answer_dict={}
        answer_correct={}
        i = i+1
        for inst in results[question][:i]:
            answer , self_certainty , verdict = inst[0], inst[1], inst[2]
            if answer not in answer_dict:
                answer_dict[answer] = 0
            
            update_value = max(1, np.exp(lambda_value * (self_certainty - predefined_confidence_mean) / (predefined_confidence_std)))
            answer_dict[answer] += update_value

            if verdict:
                correct_updates.append(self_certainty)
            else:
                wrong_updates.append(self_certainty)
            
            if answer not in answer_correct:
                answer_correct[answer] = {}
            answer_correct[answer][verdict] = answer_correct[answer].get(verdict, 0) + 1
            
        final_pred = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)[0][0]
        sorted_answer_dict = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)

        if len(sorted_answer_dict) == 1:
            a, b = sorted_answer_dict[0][1], 0
        else:
            a, b = sorted_answer_dict[0][1], sorted_answer_dict[1][1]
        a = float(a)
        b = float(b)
        prob = integrate.quad(lambda x : x**(a) * (1-x)**(b), 0.5, 1)[0] / integrate.quad(lambda x : x**(a) * (1-x)**(b), 0, 1)[0]

        if dataset == 'arcChallenge' or dataset == 'gpqa_diamond':
            majority_voting_results[question].append((final_pred.lower() == question_answer[question].lower(), prob))
        else:
            majority_voting_results[question].append((answer_correct[final_pred].get(True, 0) >= answer_correct[final_pred].get(False, 0), prob))

In [ ]:
threshold = beta_threshold
sample_size_list=[]
cnt=0
single_cnt=0
total_length = 0
for question in majority_voting_results:
    flag=False
    for i, inst in enumerate(majority_voting_results[question]):
        if i == 0:
            if results[question][0][1] >= predefined_threshold:
                sample_size_list.append(1)
                response = results[question][i][3]
                total_length += len(tokenizer.encode(response, add_special_tokens=False))
                cnt += (majority_voting_results[question][0][0] == True)
                single_cnt += (majority_voting_results[question][0][0] == True)
                flag = True
                break
        correctness, prob = inst
        response = results[question][i][3]
        total_length += len(tokenizer.encode(response, add_special_tokens=False))
        # print(f"Q: {question} | Correctness: {correctness} | Prob: {prob} at sample size {i+1}")
        if prob >= threshold:
            flag = True
            sample_size_list.append(i+1)
            cnt += (correctness == True)
            break
    if not flag:
        sample_size_list.append(len(majority_voting_results[question]))
        cnt += (majority_voting_results[question][-1][0] == True)

print(f"====================Ours ({threshold}, posterior_threshold: {posterior_threshold}, calibration size: {calibration_size})====================")
print("Ours Results:")
print("Correct:", cnt)
print("Total:", len(sample_size_list))
print("Accuracy: {:.2f}%".format(cnt/len(sample_size_list) * 100))
print("Average Sample Size:", np.mean(sample_size_list))
print("Average Response Length per Question:", total_length/len(sample_size_list))
print("Total Response Length:", total_length)
print("Flops: ", total_length * model_flops[model], "FLOPS")
print("Average TFLOPS:", (total_length * model_flops[model]) / (len(sample_size_list) * 1e12))
print("Single Sample Correct Accuracy:", single_cnt, sample_size_list.count(1), single_cnt / sample_size_list.count(1) * 100 if sample_size_list.count(1) > 0 else 0)
print("==========================================")